# Groq Reasoning Debugging Tutor

This notebook is the **Groq API** edition of the project.

Use this when you want:
- a notebook-only submission
- Groq-hosted reasoning models
- explicit reasoning controls for GPT-OSS on Groq

This version is intentionally focused:
- **Groq chat completions**
- **reasoning toggle + reasoning level**
- **few-shot prompting**
- **memory compression**


## 1. Setup and imports


In [4]:
# %pip -q install gradio openai python-dotenv

import os
import time
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

## 2. Connect to Groq


In [3]:
api_client = None
api_models = {
    "GPT-OSS 20B": "openai/gpt-oss-20b",
    "GPT-OSS 120B": "openai/gpt-oss-120b",
}

load_dotenv(".env")
api_key = os.getenv("GROQ_API_KEY")
print("GROQ_API_KEY:", "loaded" if api_key else "NOT FOUND")

if api_key:
    api_client = OpenAI(
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )
    print("Groq client ready.")
else:
    print("Add GROQ_API_KEY to your environment or .env file and rerun this cell.")

GROQ_API_KEY: loaded
Groq client ready.


## 3. Blueprint


In [6]:
app_title = "AI Debugging Tutor — Groq Reasoning"
app_desc = (
    "A notebook-only Groq debugging tutor with reasoning controls, "
    "few-shot prompting, and memory compression."
)

system_prompt = '''
You are a Socratic debugging tutor.

Rules:
- Do not give the final answer or full corrected code.
- Ask one small guiding question at a time.
- Give only the next hint or check.
- Keep answers short and clear.
- If the user asks for the answer, do not give it.
- End with one short question.
'''

reasoning_prompt = '''
Keep reasoning brief.
Always provide a short final response.
Do not spend many tokens on internal reasoning.
'''

few_shot_example = '''
Example 1:
User:
age = 25
print("I am " + age)

Assistant:
What is the type of `age` right now? Can `+` join a string and an integer directly?

Example 2:
User:
Please just give me the answer.

Assistant:
Before we jump to the answer, what part feels most confusing right now: the error message, the variable type, or the loop logic?
'''

test_cases = {
    "1. CSV Loading [Concept]": {
        "input": "How do I read a CSV file with the datascience package? Please guide me step by step."
    },
    "2. TypeError [Easy]": {
        "input": "age = 25\nprint(\"I am \" + age)\n\nTypeError: can only concatenate str to str\n\nPlease ask me one small question at a time."
    },
    "3. Average Score [Logic]": {
        "input": (
            "def average_score(scores):\n"
            "    total = 0\n"
            "    for score in scores:\n"
            "        total = score\n"
            "    return total / len(scores)\n\n"
            "print(average_score([80, 90, 70, 100]))\n\n"
            "Please help me find the logic bug step by step."
        )
    },
    "4. Prefix Sum [Index]": {
        "input": (
            "nums = [3, 1, 4, 2]\nprefix = []\ncurrent = 0\n\n"
            "for i in range(len(nums) + 1):\n"
            "    current += nums[i]\n"
            "    prefix.append(current)\n\n"
            "print(prefix)\n\n"
            "Please guide me without fixing it for me."
        )
    },
}

## 4. State


In [7]:
run_log = []
history_summary = ""

## 5. Helper functions


In [8]:
def clean_message(message):
    role = message.get("role", "user")
    content = message.get("content", "")

    if isinstance(content, str):
        text = content
    elif isinstance(content, dict):
        text = content.get("text", str(content))
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        text = "".join(parts)
    else:
        text = str(content)

    if role == "assistant":
        text = text.split("\n\n`Time: ")[0]

    return {"role": role, "content": text}


def build_summary_source(old_history):
    lines = []
    for message in old_history:
        clean_item = clean_message(message)
        lines.append(f"[{clean_item['role'].upper()}] {clean_item['content']}")
    return "\n".join(lines)


def build_prompt(system_prompt, reasoning_prompt, few_shot_example, use_reasoning, use_few_shot):
    active_prompt = system_prompt.strip()
    if use_reasoning:
        active_prompt += "\n\n" + reasoning_prompt.strip()
    if use_few_shot:
        active_prompt += "\n\n" + few_shot_example.strip()
    return active_prompt


def build_query(user_input, test_case, chat_history):
    if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
        if user_input.strip():
            return user_input.strip()
        return test_cases[test_case]["input"]
    return user_input.strip()


def split_history(chat_history, memory_turns):
    keep_messages = int(memory_turns) * 2
    if memory_turns > 0 and len(chat_history) > keep_messages + 4:
        old_history = chat_history[:-keep_messages]
        recent_history = chat_history[-keep_messages:]
    else:
        old_history = []
        recent_history = chat_history
    return old_history, recent_history


def count_text_tokens(text):
    return max(1, len(str(text).split()))


def supports_reasoning(model_name):
    model_id = api_models.get(model_name, "")
    return model_id in {"openai/gpt-oss-20b", "openai/gpt-oss-120b"}


def build_extra_body(model_name, use_reasoning, reasoning_level):
    if not supports_reasoning(model_name):
        return {}

    if not use_reasoning:
        return {"reasoning_effort": "none", "include_reasoning": False}

    selected_level = str(reasoning_level).lower()
    if selected_level not in {"low", "medium", "high"}:
        selected_level = "low"

    return {
        "reasoning_effort": selected_level,
        "include_reasoning": False,
    }


def summarize_history(old_history, model_name):
    global history_summary

    if not old_history:
        return

    if api_client is None:
        history_summary = "Groq client is not ready, so compressed history is unavailable."
        return

    try:
        response = api_client.chat.completions.create(
            model=api_models[model_name],
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Summarize this older conversation in 4 short bullet points. "
                        "Keep the user's important mistakes and useful facts. "
                        "Do not answer the user."
                    ),
                },
                {"role": "user", "content": build_summary_source(old_history)},
            ],
            temperature=0.2,
            max_completion_tokens=160,
            top_p=0.95,
        )
        history_summary = (response.choices[0].message.content or "").strip()
    except Exception as error:
        history_summary = "History compression failed: " + str(error)


def build_messages(active_prompt, chat_history, memory_turns, model_name):
    messages = [{"role": "system", "content": active_prompt}]
    old_history, recent_history = split_history(chat_history, memory_turns)

    if old_history:
        summarize_history(old_history, model_name)

    if history_summary:
        messages.append({
            "role": "system",
            "content": "Conversation summary from older turns:\n" + history_summary,
        })

    if memory_turns > 0:
        for message in recent_history:
            messages.append(clean_message(message))

    return messages


def build_context_text(messages):
    parts = []
    for message in messages:
        parts.append(f"[{message['role'].upper()}]")
        parts.append(str(message["content"]))
        parts.append("-" * 40)
    return "\n".join(parts)


def build_history_summary():
    if history_summary == "":
        return "No compressed history yet."
    return history_summary


def generate_response(messages, model_name, temperature, max_tokens, use_reasoning, reasoning_level):
    if api_client is None:
        final_text = "Groq client is not ready. Load GROQ_API_KEY and rerun the connection cell."
        return final_text, None, count_text_tokens(final_text)

    try:
        response = api_client.chat.completions.create(
            model=api_models[model_name],
            messages=messages,
            temperature=float(temperature),
            max_completion_tokens=int(max_tokens),
            top_p=0.95,
            extra_body=build_extra_body(model_name, use_reasoning, reasoning_level),
        )
        final_text = response.choices[0].message.content or ""
        usage = getattr(response, "usage", None)
        prompt_tokens = getattr(usage, "prompt_tokens", None) if usage is not None else None
        output_tokens = getattr(usage, "completion_tokens", None) if usage is not None else None
        if output_tokens is None:
            output_tokens = count_text_tokens(final_text)
        return final_text, prompt_tokens, output_tokens
    except Exception as error:
        final_text = "Generation failed: " + str(error)
        return final_text, None, count_text_tokens(final_text)


def build_log_rows(run_log):
    rows = []
    for record in run_log[-10:]:
        rows.append([
            record["run"],
            record["test case"],
            record["model"],
            record["reasoning"],
            record["input_tokens"],
            record["output_tokens"],
            record["latency"],
        ])
    return rows


def run_chatbot(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_history,
):
    global run_log

    if chat_history is None:
        chat_history = []

    active_prompt = build_prompt(
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        use_few_shot,
    )
    messages = build_messages(active_prompt, chat_history, memory_turns, model_name)
    user_query = build_query(user_input, test_case, chat_history)
    history_text = build_history_summary()

    if user_query == "":
        return chat_history, "", history_text, [], "", "**Status:** Waiting for input"

    messages.append({"role": "user", "content": user_query})
    context_text = build_context_text(messages)

    display_user = user_input.strip()
    if display_user == "":
        if len(chat_history) == 0 and test_case != "Free Typing" and test_case in test_cases:
            display_user = test_cases[test_case]["input"]
        else:
            display_user = "[" + test_case + "]"

    chat_history = list(chat_history)
    chat_history.append({"role": "user", "content": display_user})
    chat_history.append({"role": "assistant", "content": ""})

    start_time = time.perf_counter()
    final_text, api_prompt_tokens, output_tokens = generate_response(
        messages,
        model_name,
        temperature,
        max_tokens,
        use_reasoning,
        reasoning_level,
    )
    latency = round(time.perf_counter() - start_time, 2)

    input_tokens = (
        api_prompt_tokens
        if api_prompt_tokens is not None
        else count_text_tokens(context_text)
    )

    reasoning_label = (
        str(reasoning_level).lower()
        if (use_reasoning and supports_reasoning(model_name))
        else "off"
    )

    badge = (
        f"\n\n`Time: {latency}s"
        f" | Input tokens: {input_tokens}"
        f" | Output tokens: {output_tokens}"
        f" | Reasoning: {reasoning_label}`"
    )
    chat_history[-1]["content"] = final_text + badge

    run_log.append({
        "run": len(run_log) + 1,
        "test case": test_case,
        "model": model_name,
        "reasoning": reasoning_label,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency": latency,
    })

    return (
        chat_history,
        context_text,
        build_history_summary(),
        build_log_rows(run_log),
        "",
        "**Status:** Done",
    )


def clear_chat():
    global history_summary
    global run_log
    history_summary = ""
    run_log = []
    return [], "", "No compressed history yet.", [], "", "**Status:** Ready"


def load_test_case(test_case):
    if test_case == "Free Typing":
        return "", "**Status:** Ready"
    return test_cases[test_case]["input"], "**Status:** Loaded " + test_case


def run_chatbot_ui(
    user_input,
    system_prompt,
    reasoning_prompt,
    few_shot_example,
    use_reasoning,
    reasoning_level,
    use_few_shot,
    memory_turns,
    temperature,
    max_tokens,
    test_case,
    model_name,
    chat_state,
):
    chat_history, context_text, history_text, log_rows, next_input, status_message = run_chatbot(
        user_input,
        system_prompt,
        reasoning_prompt,
        few_shot_example,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        memory_turns,
        temperature,
        max_tokens,
        test_case,
        model_name,
        chat_state,
    )
    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )


def clear_chat_ui():
    chat_history, context_text, history_text, log_rows, next_input, status_message = clear_chat()
    return (
        chat_history,
        context_text,
        history_text,
        log_rows,
        next_input,
        status_message,
        chat_history,
    )

## 6. Build the interface


In [9]:
ui_test_cases = ["Free Typing"] + list(test_cases.keys())
ui_models = list(api_models.keys())

with gr.Blocks(title=app_title) as web_app:
    gr.Markdown("# " + app_title)
    gr.Markdown("*" + app_desc + "*")

    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=7):
            chat_window = gr.Chatbot(label="Chat", height=520)
            with gr.Row():
                user_input_box = gr.Textbox(
                    show_label=False,
                    placeholder="Type your debugging question here...",
                    lines=4,
                    scale=8,
                )
                with gr.Column(scale=1, min_width=60):
                    ask_button = gr.Button("Ask", variant="primary")
                    clear_button = gr.Button("New Chat")

        with gr.Column(scale=4):
            test_case_dropdown = gr.Dropdown(
                choices=ui_test_cases,
                value="Free Typing",
                label="Test Case",
            )
            model_dropdown = gr.Dropdown(
                choices=ui_models,
                value=ui_models[0] if ui_models else None,
                label="Groq Model",
            )
            use_reasoning = gr.Checkbox(label="Use Reasoning", value=True)
            reasoning_level = gr.Dropdown(
                choices=["low", "medium", "high"],
                value="low",
                label="Reasoning Level",
            )
            use_few_shot = gr.Checkbox(label="Use Few-Shot", value=False)

            with gr.Accordion("Model Settings", open=True):
                memory_turns_slider = gr.Slider(0, 8, step=1, value=1, label="Memory Turns")
                temperature_slider = gr.Slider(0.0, 1.0, step=0.1, value=0.2, label="Temperature")
                max_tokens_slider = gr.Slider(64, 2048, step=64, value=512, label="Max Tokens")

            with gr.Accordion("Edit Prompts", open=False):
                system_prompt_box = gr.Textbox(label="System Prompt", value=system_prompt, lines=8)
                reasoning_prompt_box = gr.Textbox(
                    label="Reasoning Prompt",
                    value=reasoning_prompt,
                    lines=5,
                )
                few_shot_box = gr.Textbox(label="Few-Shot Example", value=few_shot_example, lines=8)

    with gr.Row():
        context_box = gr.Textbox(label="Context", value="", lines=16, interactive=False)
        history_box = gr.Textbox(
            label="History Summary",
            value="No compressed history yet.",
            lines=16,
            interactive=False,
        )

    log_table = gr.Dataframe(
        headers=["Run", "Test Case", "Model", "Reasoning", "Input", "Output", "Latency"],
        datatype=["number", "str", "str", "str", "number", "number", "number"],
        row_count=10,
        column_count=(7, "fixed"),
        interactive=False,
        label="Experiment Log",
    )

    status_box = gr.Markdown("**Status:** Ready")

    inputs = [
        user_input_box,
        system_prompt_box,
        reasoning_prompt_box,
        few_shot_box,
        use_reasoning,
        reasoning_level,
        use_few_shot,
        memory_turns_slider,
        temperature_slider,
        max_tokens_slider,
        test_case_dropdown,
        model_dropdown,
        chat_state,
    ]

    outputs = [
        chat_window,
        context_box,
        history_box,
        log_table,
        user_input_box,
        status_box,
        chat_state,
    ]

    test_case_dropdown.change(load_test_case, test_case_dropdown, [user_input_box, status_box])
    ask_button.click(run_chatbot_ui, inputs, outputs)
    user_input_box.submit(run_chatbot_ui, inputs, outputs)
    clear_button.click(clear_chat_ui, None, outputs)

## 7. Run the app


In [10]:
web_app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7866
* Running on public URL: https://ed32745ff7b6d13619.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
